# ĐỒ ÁN 1: MA TRẬN VÀ CỞ SỞ CỦA TÍNH TOÁN KHOA HỌC
## PHẦN 1: PHÉP KHỬ GAUSS VÀ CÁC ỨNG DỤNG

**Trường:** Đại học Khoa học Tự nhiên - ĐHQG HCM
**Môn học:** Toán Ứng Dụng và Thống Kê  

---
**Tóm tắt:** Notebook này trình bày các kết quả thử nghiệm của việc cài đặt thuật toán khử Gauss có chọn phần tử chốt (Partial Pivoting) từ đầu (from scratch) bằng Python. Các module được tách thành 4 file `.py` riêng biệt và được gọi vào đây để đối chiếu kết quả với thư viện `NumPy`.

In [3]:
import numpy as np

# Import các module của nhóm
from gaussian import gaussian_eliminate, general_solution, verify_solution
from determinant import determinant, verify_determinant
from inverse import inverse
from rank_basis import rank_and_basis

np.set_printoptions(precision=4, suppress=True, linewidth=100)
print("✅ Đã load thành công thư viện và các module của Nhóm 10!")

✅ Đã load thành công thư viện và các module của Nhóm 10!


## 1. Giải hệ phương trình tuyến tính (Gauss Elimination)
Kiểm tra khả năng giải hệ phương trình $Ax = b$ trong các trường hợp: nghiệm duy nhất, vô số nghiệm và kết hợp đối chiếu sai số với NumPy.

In [9]:
np.random.seed(42)

def create_random_system(m, n, rank=None):
    """Tạo hệ Ax = b với nghiệm x đã biết"""
    if rank is None:
        rank = min(m, n)
    else:
        rank = min(rank, m, n)
    
    # Tạo ma trận A ngẫu nhiên
    A = np.random.randn(m, n)
    
    # Đảm bảo rank mong muốn
    if rank < min(m, n):
        U, S, Vt = np.linalg.svd(A)
        S[rank:] = 0
        A = U @ np.diag(S) @ Vt
    
    # Tạo nghiệm x ngẫu nhiên
    x_true = np.random.randn(n)
    
    # Tính b
    b = A @ x_true
    
    return A.tolist(), b.tolist(), x_true.tolist()

test_cases = [
    ("Hệ có nghiệm duy nhất", 
     [[2, 1, -1], [-3, -1, 2], [-2, 1, 2]], 
     [8, -11, -3]),
    
    ("A[0][0] = 0, cần Partial Pivoting", 
     [[0, 2, 1], [1, -2, -3], [-1, 1, 2]], 
     [-8, 0, 3]),
    
    ("Overdetermined có nghiệm", 
     [[1, 1], [2, -1], [3, 0]], 
     [3, 0, 3]),
    
    ("Hệ vô nghiệm", 
     [[1, 1, 1], [2, 2, 2], [1, -1, 1]], 
     [6, 10, 2]),
    
    ("Overdetermined vô nghiệm", 
     [[2, 1, -1], [1, -1, 2], [3, 2, -1], [1, 3, 1]], 
     [4, 5, 7, 10]),
    
    ("Hệ vô số nghiệm", 
     [[1, 1, 1], [2, 2, 2], [1, -1, 1]], 
     [6, 12, 2]),
    
    ("Underdetermined", 
     [[1, 1, 1], [1, -1, 1]], 
     [6, 2]),
    
    ("Ma trận Hilbert 4x4", 
     [[1, 1/2, 1/3, 1/4],
      [1/2, 1/3, 1/4, 1/5],
      [1/3, 1/4, 1/5, 1/6],
      [1/4, 1/5, 1/6, 1/7]],
     [1 + 0.5 + 1/3 + 1/4,
      0.5 + 1/3 + 1/4 + 0.2,
      1/3 + 1/4 + 0.2 + 1/6,
      1/4 + 0.2 + 1/6 + 1/7]),
    
    ("Ma trận gần suy biến", 
     [[1, 1, 1, 1, 1],
      [1, 1, 1, 1, 2],
      [1, 1, 1, 2, 2],
      [1, 1, 2, 2, 2],
      [1, 2, 2, 2, 2]],
     [15, 20, 24, 27, 29]),
    
    ("Ma trận có pivot rất nhỏ", 
     [[1e-10, 1, 0, 0, 0],
      [1, 1e-10, 1, 0, 0],
      [0, 1, 1e-10, 1, 0],
      [0, 0, 1, 1e-10, 1],
      [0, 0, 0, 1, 1e-10]],
     [1e-10 + 1, 1 + 1e-10, 1e-10 + 1, 1 + 1e-10, 1e-10 + 1])
]

random_cases = [
    (50, 50, None, "Ma trận vuông ngẫu nhiên 50x50"),
    (50, 100, None, "Ma trận ngẫu nhiên 50x100"),
    (80, 40, 40, "Ma trận ngẫu nhiên 80x40 (rank đầy đủ)"),
    (60, 60, 45, "Ma trận ngẫu nhiên 60x60 (rank suy biến)")
]

for m, n, rank, desc in random_cases:
    A_rand, b_rand, x_true_rand = create_random_system(m, n, rank)
    test_cases.append((desc, A_rand, b_rand))

for i, (desc, A, b) in enumerate(test_cases, 1):
    print(f"\n--- Test {i}: {desc} ---")
    print(f"Kích thước ma trận: {len(A)} x {len(A[0]) if A else 0}")

    result = gaussian_eliminate(A, b)
    
    if result is None or len(result) < 2:
        print("Lỗi: Hàm gaussian_eliminate không trả về kết quả hợp lệ")
        continue
        
    _, x, _ = result
    
    # Nếu có nghiệm
    if x is not None and len(x) > 0:
        print(f"Nghiệm tìm được (10 giá trị đầu): {[round(val, 6) for val in x[:10]]}")
        if len(x) > 10:
            print(f"... và {len(x) - 10} giá trị khác")
        
        verification_result = verify_solution(A, x, b)
        
        if verification_result == 0:
            print("Kết luận: NGHIỆM CHÍNH XÁC")
        elif verification_result == 1:
            print("Kết luận: NGHIỆM GẦN ĐÚNG")
        else:
            print("Kết luận: NGHIỆM SAI LỆCH LỚN")
            
    else:
        # Trường hợp vô nghiệm hoặc vô số nghiệm
        try:
            particular, nullspace, free_vars = general_solution(A, b)
            if particular is None and nullspace is None:
                print("Kết luận: HỆ VÔ NGHIỆM")
            else:
                print("Kết luận: HỆ VÔ SỐ NGHIỆM")
                if particular:
                    print(f"Nghiệm riêng (10 giá trị đầu): {[round(val, 6) for val in particular[:10]]}")
                if free_vars and len(free_vars) > 0:
                    var_names = [f"x{v+1}" for v in free_vars[:10]]
                    print(f"Biến tự do: {', '.join(var_names)}")
        except Exception as e:
            print(f"Không thể xác định loại nghiệm: {e}")


--- Test 1: Hệ có nghiệm duy nhất ---
Kích thước ma trận: 3 x 3
Nghiệm tìm được (10 giá trị đầu): [2.0, 3.0, -1.0]
  -> [NumPy Verify] Residual norm ||Ax - b||_inf: 8.88e-16
  -> [NumPy Verify] Kết quả: NGHIỆM CHÍNH XÁC
Kết luận: NGHIỆM CHÍNH XÁC

--- Test 2: A[0][0] = 0, cần Partial Pivoting ---
Kích thước ma trận: 3 x 3
Nghiệm tìm được (10 giá trị đầu): [-4.0, -5.0, 2.0]
  -> [NumPy Verify] Residual norm ||Ax - b||_inf: 0.00e+00
  -> [NumPy Verify] Kết quả: NGHIỆM CHÍNH XÁC
Kết luận: NGHIỆM CHÍNH XÁC

--- Test 3: Overdetermined có nghiệm ---
Kích thước ma trận: 3 x 2
Nghiệm tìm được (10 giá trị đầu): [1.0, 2.0]
  -> [NumPy Verify] Residual norm ||Ax - b||_inf: 0.00e+00
  -> [NumPy Verify] Kết quả: NGHIỆM CHÍNH XÁC
Kết luận: NGHIỆM CHÍNH XÁC

--- Test 4: Hệ vô nghiệm ---
Kích thước ma trận: 3 x 3
Kết luận: HỆ VÔ NGHIỆM

--- Test 5: Overdetermined vô nghiệm ---
Kích thước ma trận: 4 x 3
Kết luận: HỆ VÔ NGHIỆM

--- Test 6: Hệ vô số nghiệm ---
Kích thước ma trận: 3 x 3
Kết luận: HỆ VÔ S

## 2. Tính định thức (Determinant)
Kiểm tra thuật toán tính định thức dựa trên số lần hoán đổi dòng ($s$) trong phép khử Gauss.

In [8]:
np.random.seed(42)

def create_random_matrix(n, cond_number=None):
    """Tạo ma trận vuông ngẫu nhiên với số điều kiện tùy chọn"""
    A = np.random.randn(n, n)
    
    if cond_number is not None:
        # Điều chỉnh số điều kiện
        U, S, Vt = np.linalg.svd(A)
        S = np.linspace(cond_number, 1, n)  # Tạo các singular values
        A = U @ np.diag(S) @ Vt
    
    return A.tolist()

test_cases = [
    ("Ma trận tam giác trên",
     [[1, 2, 3, 4],
      [0, 5, 6, 7],
      [0, 0, 8, 9],
      [0, 0, 0, 10]]),
    
    ("Ma trận suy biến (2 dòng giống nhau)",
     [[1, 2, 3, 4],
      [5, 6, 7, 8],
      [1, 2, 3, 4],
      [9, 10, 11, 12]]),
    
    ("Ma trận Hilbert 4x4 (ill-conditioned)",
     [[1, 1/2, 1/3, 1/4],
      [1/2, 1/3, 1/4, 1/5],
      [1/3, 1/4, 1/5, 1/6],
      [1/4, 1/5, 1/6, 1/7]]),
    
    ("Ma trận không vuông (bắt lỗi)",
     [[1, 2, 3], 
      [4, 5, 6]]),
    
    ("Ma trận gần suy biến",
     [[1, 1, 1, 1, 1, 1],
      [1, 1, 1, 1, 1, 2],
      [1, 1, 1, 1, 2, 2],
      [1, 1, 1, 2, 2, 2],
      [1, 1, 2, 2, 2, 2],
      [1, 2, 2, 2, 2, 2]]),
    
    ("Ma trận có pivot nhỏ",
     [[1e-10, 1, 0, 0, 0],
      [1, 1e-10, 1, 0, 0],
      [0, 1, 1e-10, 1, 0],
      [0, 0, 1, 1e-10, 1],
      [0, 0, 0, 1, 1e-10]]),
]

random_sizes = [20, 50, 70, 90, 100]

for n in random_sizes:
    A_rand = create_random_matrix(n)
    test_cases.append((f"Ma trận ngẫu nhiên {n}x{n}", A_rand))

for n, cond in [(20, 1e3), (30, 1e6), (40, 1e8)]:
    A_rand_ill = create_random_matrix(n, cond)
    test_cases.append((f"Ma trận ngẫu nhiên {n}x{n} (số điều kiện ~{cond:.0e})", A_rand_ill))

for i, (title, A) in enumerate(test_cases, 1):
    print(f"\n--- Test {i}: {title} ---")
    print(f"Kích thước ma trận: {len(A)} x {len(A[0]) if A else 0}")
    
    try:
        det = determinant(A)
        
        if det is not None:
            print(f"Định thức tính được: {det:.6e}")
            
            try:
                A_np = np.array(A, dtype=float)
                det_np = np.linalg.det(A_np)
                print(f"Định thức numpy: {det_np:.6e}")
                
                # Tính sai số tương đối
                if abs(det_np) > 1e-10:
                    rel_error = abs(det - det_np) / abs(det_np)
                    abs_error = abs(det - det_np)
                    print(f"Sai số tuyệt đối: {abs_error:.2e}")
                    print(f"Sai số tương đối: {rel_error:.2e}")
                    
                    # Đánh giá kết quả
                    if rel_error < 1e-10 or abs_error < 1e-10:
                        print("  -> Kết luận: ĐỊNH THỨC ĐÚNG")
                    elif rel_error < 1e-6 or abs_error < 1e-6:
                        print("  -> Kết luận: ĐỊNH THỨC GẦN ĐÚNG")
                    else:
                        print("  -> Kết luận: ĐỊNH THỨC SAI")
                else:
                    # Định thức numpy rất nhỏ (gần 0)
                    if abs(det) < 1e-10:
                        print("  -> Kết luận: ĐỊNH THỨC ĐÚNG (det ≈ 0)")
                    else:
                        print("  -> Kết luận: ĐỊNH THỨC GẦN ĐÚNG (det numpy ≈ 0)")
                        
            except np.linalg.LinAlgError as e:
                print(f"Lỗi numpy: {e}")
                print("  -> Kết luận: KHÔNG THỂ SO SÁNH")
            except Exception as e:
                print(f"Lỗi không xác định: {e}")
                print("  -> Kết luận: LỖI KHI KIỂM TRA")
        else:
            print("Định thức tính được: None (có thể do ma trận không vuông hoặc lỗi)")
            print("  -> Kết luận: KHÔNG TÍNH ĐƯỢC ĐỊNH THỨC")
            
    except Exception as e:
        print(f"Lỗi khi tính định thức: {e}")
        print("  -> Kết luận: LỖI THUẬT TOÁN")


--- Test 1: Ma trận tam giác trên ---
Kích thước ma trận: 4 x 4
Định thức tính được: 4.000000e+02
Định thức numpy: 4.000000e+02
Sai số tuyệt đối: 1.14e-13
Sai số tương đối: 2.84e-16
  -> Kết luận: ĐỊNH THỨC ĐÚNG

--- Test 2: Ma trận suy biến (2 dòng giống nhau) ---
Kích thước ma trận: 4 x 4
Định thức tính được: 0.000000e+00
Định thức numpy: 0.000000e+00
  -> Kết luận: ĐỊNH THỨC ĐÚNG (det ≈ 0)

--- Test 3: Ma trận Hilbert 4x4 (ill-conditioned) ---
Kích thước ma trận: 4 x 4
Định thức tính được: 1.653439e-07
Định thức numpy: 1.653439e-07
Sai số tuyệt đối: 2.59e-20
Sai số tương đối: 1.57e-13
  -> Kết luận: ĐỊNH THỨC ĐÚNG

--- Test 4: Ma trận không vuông (bắt lỗi) ---
Kích thước ma trận: 2 x 3
Định thức tính được: None (có thể do ma trận không vuông hoặc lỗi)
  -> Kết luận: KHÔNG TÍNH ĐƯỢC ĐỊNH THỨC

--- Test 5: Ma trận gần suy biến ---
Kích thước ma trận: 6 x 6
Định thức tính được: 1.000000e+00
Định thức numpy: 1.000000e+00
Sai số tuyệt đối: 0.00e+00
Sai số tương đối: 0.00e+00
  -> Kết lu

## 3. Tìm ma trận nghịch đảo (Inverse Matrix)
Sử dụng thuật toán Gauss-Jordan trên ma trận mở rộng $[A | I]$. Để kiểm chứng, ta sẽ nhân ma trận gốc $A$ với ma trận nghịch đảo $A^{-1}$ tìm được để xem có ra ma trận đơn vị $I$ hay không.

In [4]:
np.random.seed(42)

cases_inv = [
    ("Ma trận 3x3 khả nghịch", [[2, 1, 1], [1, 3, 2], [1, 0, 0]]),
    ("Ma trận có phần tử A[0][0] = 0", [[0, 2, 3], [1, 1, -1], [2, -1, 1]]),
    ("Ma trận suy biến (det = 0)", [[1, 2, 3], [4, 5, 6], [7, 8, 9]]),
    ("Ma trận chứa số cực nhỏ", [[1e-10, 1], [1, 1]]),
    ("Ma trận không vuông (Bắt lỗi)", [[1, 2, 3], [4, 5, 6]]),
    ("Ma trận ngẫu nhiên (50x50)", np.random.rand(50, 50).tolist())
]

for i, (desc, A) in enumerate(cases_inv, 1):
    print(f"\n--- Test {i}: {desc} ---")
    invA = inverse(A)
    
    if invA is not None:
        A_np, invA_np = np.array(A), np.array(invA)
        
        err_identity = float(np.linalg.norm((A_np @ invA_np) - np.eye(len(A)), ord=np.inf))
        err_numpy = float(np.linalg.norm(invA_np - np.linalg.inv(A_np), ord=np.inf))
        
        print(f"Sai số ||A * A^-1 - I||_inf : {err_identity:.2e}")
        print(f"Sai số ||A^-1 - NumPy||_inf : {err_numpy:.2e}")
    else:
        print("Kết quả: Không tồn tại ma trận nghịch đảo.")


--- Test 1: Ma trận 3x3 khả nghịch ---
Sai số ||A * A^-1 - I||_inf : 8.88e-16
Sai số ||A^-1 - NumPy||_inf : 2.00e-15

--- Test 2: Ma trận có phần tử A[0][0] = 0 ---
Sai số ||A * A^-1 - I||_inf : 1.11e-16
Sai số ||A^-1 - NumPy||_inf : 8.33e-17

--- Test 3: Ma trận suy biến (det = 0) ---
Lỗi: Ma trận suy biến (Singular Matrix), không thể tìm nghịch đảo.
Kết quả: Không tồn tại ma trận nghịch đảo.

--- Test 4: Ma trận chứa số cực nhỏ ---
Sai số ||A * A^-1 - I||_inf : 0.00e+00
Sai số ||A^-1 - NumPy||_inf : 0.00e+00

--- Test 5: Ma trận không vuông (Bắt lỗi) ---
Lỗi: Ma trận không vuông, không thể tìm nghịch đảo.
Kết quả: Không tồn tại ma trận nghịch đảo.

--- Test 6: Ma trận ngẫu nhiên (50x50) ---
Sai số ||A * A^-1 - I||_inf : 7.17e-14
Sai số ||A^-1 - NumPy||_inf : 2.38e-13


## 4. Hạng và cơ sở (Rank & Basis)
Từ ma trận bậc thang rút gọn (RREF), trích xuất hạng của ma trận và cơ sở của 3 không gian vector (Cột, Dòng, Nghiệm). So sánh hạng với hàm `matrix_rank` của NumPy.

In [5]:
np.random.seed(42)

cases_rank = [
    ("Ma trận 3x4 (Hạng 2)", [[1, 2, 0, 1], [2, 4, 1, 4], [3, 6, 1, 5]]),
    ("Ma trận vuông khả nghịch", [[2, 1, 1], [1, 3, 2], [1, 0, 0]]),
    ("Ma trận toàn số 0", [[0, 0, 0], [0, 0, 0]]),
    ("Ma trận có các dòng tỷ lệ (Hạng 1)", [[1, 2, 3], [2, 4, 6], [3, 6, 9]]),
    ("Ma trận đứng (4x2)", [[1, 2], [3, 4], [5, 6], [7, 8]]),
    ("Ma trận ngẫu nhiên chữ nhật (50x70)", np.random.rand(50, 70).tolist())
]

for i, (desc, A) in enumerate(cases_rank, 1):
    print(f"\n--- Test {i}: {desc} ---")
    res = rank_and_basis(A)
    
    A_np = np.array(A)
    rank_np = np.linalg.matrix_rank(A_np)
    
    n_cols = len(A[0]) if A else 0
    dim_col = len(res['column_space_basis'])
    dim_row = len(res['row_space_basis'])
    dim_null = len(res['null_space_basis'])
    
    print(f"Hạng (Tính toán / NumPy) : {res['rank']} / {rank_np}")
    print(f"Số chiều không gian      : Cột = {dim_col}, Dòng = {dim_row}, Nghiệm = {dim_null}")
    
    if dim_col + dim_null == n_cols:
        print(f"Định lý Rank-Nullity     : Hợp lệ ({dim_col} + {dim_null} = {n_cols})")
    else:
        print(f"Định lý Rank-Nullity     : Có sai sót!")


--- Test 1: Ma trận 3x4 (Hạng 2) ---
Hạng (Tính toán / NumPy) : 2 / 2
Số chiều không gian      : Cột = 2, Dòng = 2, Nghiệm = 2
Định lý Rank-Nullity     : Hợp lệ (2 + 2 = 4)

--- Test 2: Ma trận vuông khả nghịch ---
Hạng (Tính toán / NumPy) : 3 / 3
Số chiều không gian      : Cột = 3, Dòng = 3, Nghiệm = 0
Định lý Rank-Nullity     : Hợp lệ (3 + 0 = 3)

--- Test 3: Ma trận toàn số 0 ---
Hạng (Tính toán / NumPy) : 0 / 0
Số chiều không gian      : Cột = 0, Dòng = 0, Nghiệm = 3
Định lý Rank-Nullity     : Hợp lệ (0 + 3 = 3)

--- Test 4: Ma trận có các dòng tỷ lệ (Hạng 1) ---
Hạng (Tính toán / NumPy) : 1 / 1
Số chiều không gian      : Cột = 1, Dòng = 1, Nghiệm = 2
Định lý Rank-Nullity     : Hợp lệ (1 + 2 = 3)

--- Test 5: Ma trận đứng (4x2) ---
Hạng (Tính toán / NumPy) : 2 / 2
Số chiều không gian      : Cột = 2, Dòng = 2, Nghiệm = 0
Định lý Rank-Nullity     : Hợp lệ (2 + 0 = 2)

--- Test 6: Ma trận ngẫu nhiên chữ nhật (50x70) ---
Hạng (Tính toán / NumPy) : 50 / 50
Số chiều không gian      : Cộ